**Description:**

Here

# Imports

In [15]:
import os
import time
import requests
import pandas as pd

In [22]:
from dotenv import load_dotenv
load_dotenv()
RPC_API_KEY = os.getenv("RPC_API_KEY")
print("RPC_API_KEY loaded:", RPC_API_KEY is not None)

RPC_API_KEY loaded: True


# Download trade information from Alchemy

## Dataset import and cleaning

In [16]:
balancer_v2_rLBPs = pd.read_csv("../../../media/dune_web_downloads/rLBPs/rLBPs_BalancerV2_Ethereum_pools_addresses.csv", dtype={'poolId': str, 'project_token_address': str})
balancer_v2_rLBPs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 13 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   pool_address                       12 non-null     object 
 1   poolId                             12 non-null     object 
 2   collateral_token                   12 non-null     object 
 3   collateral_symbol                  12 non-null     object 
 4   collateral_start_weight            12 non-null     float64
 5   collateral_end_weight              12 non-null     float64
 6   collateral_initial_balance_raw     12 non-null     float64
 7   project_token_address              12 non-null     object 
 8   project_token_initial_balance_raw  12 non-null     float64
 9   start_block_approx                 12 non-null     int64  
 10  end_block_est                      12 non-null     int64  
 11  startTime                          12 non-null     int64  
 

In [17]:
balancer_v2_rLBPs['poolId'].nunique()

6

In [18]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
OUTPUT_FILE = '../../../media/dune_web_downloads/rLBPs/dev_rLBPs_cleaned_dune_data.csv'

# Known Decimals map
DECIMALS = {
    'USDC': 6,
    'USDT': 6,
    'DAI': 18,
    'WETH': 18,
    'BAL': 18,
    'MIM': 18,
    'FRAX': 18,
    'LUSD': 18,
    'DEFAULT': 18 
}


print("Loading raw data...")
# Read CSV (Ensure poolId is string to prevent truncation)

# Normalize IDs
balancer_v2_rLBPs['poolId'] = balancer_v2_rLBPs['poolId'].str.lower()
balancer_v2_rLBPs['pool_address'] = balancer_v2_rLBPs['pool_address'].str.lower()

print(f"Original Row Count: {len(balancer_v2_rLBPs)}")

# --- STEP 1: IDENTIFY MAIN SALE (Deduplication) ---
# We define the "Main Sale" as the event with the largest change in weights.
# Small changes (e.g. 50 -> 49.5) are usually pauses/reschedules.
# Large changes (e.g. 80 -> 20) are the actual LBP.

balancer_v2_rLBPs['weight_diff'] = abs(balancer_v2_rLBPs['collateral_start_weight'] - balancer_v2_rLBPs['collateral_end_weight'])

# Sort by Pool and Weight Difference (Largest first)
df_sorted = balancer_v2_rLBPs.sort_values(by=['pool_address', 'weight_diff'], ascending=[True, False])

# Keep only the top row for each pool
df_clean = df_sorted.drop_duplicates(subset='pool_address', keep='first').copy()

print(f"Cleaned Row Count: {len(df_clean)} (Removed {len(balancer_v2_rLBPs) - len(df_clean)} duplicate schedules)")

# --- STEP 2: CONVERT RAW BALANCES ---
# Helper to apply decimal conversion
def convert_balance(row, col_name, is_collateral=False):
    raw_val = row[col_name]
    
    # Handle empty/NaN
    if pd.isna(raw_val) or raw_val == '':
        return 0.0
        
    # Determine decimals
    if is_collateral:
        symbol = row.get('collateral_symbol', 'DEFAULT')
        decimals = DECIMALS.get(symbol, 18)
    else:
        # For project tokens, we assume 18 usually, unless we query a token list
        decimals = 18 
        
    try:
        # Convert scientific notation string (e.g., "4e+23") to float
        val = float(raw_val)
        return val / (10 ** decimals)
    except (ValueError, TypeError):
        return 0.0

# Apply conversion
df_clean['collateral_balance'] = df_clean.apply(
    lambda x: convert_balance(x, 'collateral_initial_balance_raw', is_collateral=True), axis=1
)

df_clean['project_balance'] = df_clean.apply(
    lambda x: convert_balance(x, 'project_token_initial_balance_raw', is_collateral=False), axis=1
)

# --- STEP 3: FORMAT DATES ---
df_clean['start_date'] = pd.to_datetime(df_clean['startTime'], unit='s')
df_clean['end_date'] = pd.to_datetime(df_clean['endTime'], unit='s')

# Calculate Duration (in Days)
df_clean['duration_days'] = (df_clean['endTime'] - df_clean['startTime']) / 86400

# --- STEP 4: FINAL CLEANUP ---
# Select and reorder useful columns
final_cols = [
    'pool_address', 'poolId', 'collateral_symbol', 
    'collateral_start_weight', 'collateral_end_weight', 
    'collateral_balance', 'project_balance',
    'start_block_approx', 'end_block_est', 
    'start_date', 'end_date', 'duration_days',
    'project_token_address'
]

df_final = df_clean[final_cols]

# Save
df_final.to_csv(OUTPUT_FILE, index=False)
print(f"\nSuccess! Saved cleaned data to {OUTPUT_FILE}")
print(df_final[['collateral_symbol', 'collateral_balance', 'project_balance']].head())

Loading raw data...
Original Row Count: 12
Cleaned Row Count: 6 (Removed 6 duplicate schedules)

Success! Saved cleaned data to ../../../media/dune_web_downloads/rLBPs/dev_rLBPs_cleaned_dune_data.csv
   collateral_symbol  collateral_balance  project_balance
0                DAI       400000.000000    120000.000000
1                DAI       250000.000000     76000.000000
4                DAI        40000.000000     13500.000000
2                DAI        77000.000000     32000.000000
10              WETH         1614.228205       203.046386


In [ ]:
OUTPUT_FILE = '../../../media/dune_web_downloads/rLBPs/dev_rLBPs_temporal_trades.csv'

# API SETUP
RPC_URL = f"https://eth-mainnet.g.alchemy.com/v2/{RPC_API_KEY}"
VAULT   = "0xBA12222222228d8Ba445958a75a0704d566BF2C8"

# ==========================================
# 2. FETCHER FUNCTION (Micro-Chunks)
# ==========================================
def get_trades_for_pool(pool_id, start_block, end_block, pool_name="Unknown"):
    print(f"\n[Pool: {pool_name}] Fetching {start_block} -> {end_block}...")
    
    # Topics
    swap_topic = "0x2170c741c41531aec20e7c107c24eecfdd15e69c9bb0a8dd37b1840b9e0b207b"
    pool_topic = pool_id if pool_id.startswith("0x") else "0x" + pool_id

    pool_logs = []
    chunk_size = 10  # Free Tier Limit
    
    total_blocks = end_block - start_block
    processed = 0

    for current_start in range(start_block, end_block, chunk_size):
        current_end = min(current_start + chunk_size - 1, end_block)
        
        # Payload
        payload = {
            "id": 1, "jsonrpc": "2.0", "method": "eth_getLogs",
            "params": [{
                "address": VAULT,
                "fromBlock": hex(current_start),
                "toBlock": hex(current_end),
                "topics": [swap_topic, pool_topic]
            }]
        }
        
        # Retry Loop (Robustness)
        retries = 3
        while retries > 0:
            try:
                resp = requests.post(RPC_URL, json=payload, timeout=10)
                resp.raise_for_status()
                data = resp.json()
                
                if 'error' in data:
                    print(f"  ! RPC Error: {data['error']['message']}")
                    time.sleep(2)
                    retries -= 1
                    continue
                
                # Success
                logs = data.get('result', [])
                for log in logs:
                    # Append raw data immediately to list
                    pool_logs.append({
                        'pool_id': pool_id,
                        'pool_name': pool_name,
                        'block_number': int(log['blockNumber'], 16),
                        'tx_hash': log['transactionHash'],
                        'log_index': int(log.get('logIndex', '0x0'), 16),
                        'topic1_token_in': log['topics'][1],  # Raw hex, decode later
                        'topic2_token_out': log['topics'][2], # Raw hex, decode later
                        'data': log['data'] # Contains amounts, decode later
                    })
                break # Break retry loop on success
                
            except Exception as e:
                print(f"  ! Connection Error: {e}")
                time.sleep(1)
                retries -= 1
        
        # Progress Bar logic
        processed += chunk_size
        if processed % 1000 == 0 or processed >= total_blocks:
            pct = (processed / total_blocks) * 100
            print(f"  > Progress: {pct:.1f}% ({len(pool_logs)} trades found)", end="\r")
            
        # Rate Limit Sleep
        time.sleep(0.05) 

    print(f"\n  Done. Total trades for pool: {len(pool_logs)}")
    return pool_logs

file_exists = os.path.isfile(OUTPUT_FILE)
if not file_exists:
    # Create empty DataFrame with columns to initialize file
    pd.DataFrame(columns=[
        'poolId', 'pool_name', 'block_number', 'tx_hash', 
        'log_index', 'topic1_token_in', 'topic2_token_out', 'data'
    ]).to_csv(OUTPUT_FILE, index=False)

# 3. Loop through Pools
for index, row in df_final.iterrows():
    # EXTRACT PARAMETERS (Adjust column names if your CSV is different)
    pid   = row['poolId']
    start = int(row['start_block_approx'])
    end   = int(row['end_block_est'])
    name  = f"{row['collateral_symbol']}-LBP" # Helper name for logging

    existing_df = pd.read_csv(OUTPUT_FILE)
    if pid in existing_df['poolId'].values:
        print(f"Skipping {name} (Already in output)")
        continue
    
    trades = get_trades_for_pool(pid, start, end, name)
    
    if trades:
        balancer_v2_rLBPs_trades = pd.DataFrame(trades)
        balancer_v2_rLBPs_trades.to_csv(OUTPUT_FILE, mode='a', header=False, index=False)
        print(f"  Saved {len(trades)} rows to {OUTPUT_FILE}")
    else:
        print(f"  No trades found for {name}.")

print("\nAll pools processed!")

Skipping DAI-LBP (Already in output)
Skipping DAI-LBP (Already in output)
Skipping DAI-LBP (Already in output)
Skipping DAI-LBP (Already in output)

[Pool: WETH-LBP] Fetching 13642215 -> 13515457...

  Done. Total trades for pool: 0
  No trades found for WETH-LBP.
Skipping DAI-LBP (Already in output)

All pools processed!
